<a href="https://colab.research.google.com/github/kirusha20220932-cm/CRM-CW1_20220932/blob/main/CRM_CM_BI_CW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install yfinance

import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
import datetime

# library versions used
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("yfinance:", yf.__version__)

pandas: 2.2.3
numpy: 2.4.6
yfinance: 0.2.66


In [2]:
# Part 1: get the S&P 500 list from Wikipedia
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
table = pd.read_html(url, storage_options={"User-Agent": "Mozilla/5.0"})
sp500_df = table[0]

retrieval_date = datetime.date.today()
print("Retrieval date:", retrieval_date)

# yahoo uses '-' instead of '.' (BRK.B -> BRK-B)
sp500_df['Ticker'] = sp500_df['Symbol'].str.replace('.', '-', regex=False)
tickers = sp500_df['Ticker'].tolist()
print("Number of tickers:", len(tickers))

# save a copy of the list
sp500_df.to_csv('sp500_tickers.csv', index=False)
sp500_df[['Symbol', 'Security', 'GICS Sector']].head()

Retrieval date: 2026-11-20
Number of tickers: 503


,Symbol,Security,GICS Sector
0,MMM,3M,Industrials
1,AOS,A. O. Smith,Industrials
2,ABT,Abbott Laboratories,Health Care
3,ABBV,AbbVie,Health Care
4,ACN,Accenture,Information Technology


In [3]:
# Part 2: download daily data for all shares and the S&P 500 index
start = '2021-01-01'
end = '2025-01-01'     # end date not included, so last day is 31 Dec 2024

data = yf.download(tickers + ['^GSPC'], start=start, end=end,
                   auto_adjust=False, actions=True)

df_close = data['Close']
df_adj = data['Adj Close']
df_volume = data['Volume']
df_splits = data['Stock Splits']
df_dividends = data['Dividends']

print("Shape (days, tickers):", df_adj.shape)
print("First date:", df_adj.index.min().date(), "| Last date:", df_adj.index.max().date())

# tickers that didn't download anything
failed = [t for t in df_adj.columns if df_adj[t].isna().all()]
print("Tickers with no data:", failed)

# save the raw data so I don't have to download again
df_close.to_csv('close.csv')
df_adj.to_csv('adj_close.csv')
df_volume.to_csv('volume.csv')
df_splits.to_csv('splits.csv')
df_dividends.to_csv('dividends.csv')

[*********************100%***********************]  504 of 504 completed
ERROR:yfinance:
5 Failed downloads:
ERROR:yfinance:['Q', 'HONA', 'FDXF', 'VYLR', 'SNDK']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


Shape (days, tickers): (1005, 504)
First date: 2021-01-04 | Last date: 2024-12-31
Tickers with no data: ['FDXF', 'HONA', 'Q', 'SNDK', 'VYLR']
